# Polymarket Data Infrastructure - Full Capabilities Showcase

This notebook demonstrates the complete functionality of the CUIC Quant Polymarket data infrastructure.

## Features
1. **Live API Access** - Fetch real-time data from Polymarket
2. **Database Storage** - Query historical data from local SQLite
3. **Order Book Data** - View bid/ask spreads
4. **Search & Filter** - Find markets by keyword
5. **Data Collection** - Trigger collection from notebook

---

## 1. Setup & Imports

In [ ]:
# Core imports
from cuic_quant.notebook import pm
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 150)
pd.set_option('display.max_rows', 20)

print("Imports successful")
print(f"\nAvailable methods on 'pm':")
print("  Live API:    pm.fetch_markets(), pm.fetch_orderbook()")
print("  Database:    pm.load_markets(), pm.load_top_markets(), pm.search()")
print("  Collection:  pm.collect_now(), pm.stats()")

## 2. Database Statistics

Check what data is currently stored in the local database.

In [ ]:
stats = pm.stats()

print("=" * 50)
print("DATABASE STATISTICS")
print("=" * 50)
for key, value in stats.items():
    print(f"  {key:25} : {value}")

## 3. Live API - Fetch Markets

Fetch real-time market data directly from the Polymarket API.

In [ ]:
# Fetch live markets from Polymarket API
live_df = pm.fetch_markets(limit=20, active=True)

print(f"Fetched {len(live_df)} markets from Polymarket API")
print()
live_df[['question', 'yes_price', 'no_price', 'volume', 'status']].head(10)

## 4. Order Book Data

Fetch the full order book (bids and asks) for a specific market token.

**Note:** Order books are only available for active markets with liquidity.

In [ ]:
# Get a market with a token ID
df = pm.load_markets(limit=100, active_only=False)
markets_with_tokens = df[df['yes_token_id'].notna()]

if len(markets_with_tokens) > 0:
    # Pick first market with a token
    sample_market = markets_with_tokens.iloc[0]
    token_id = sample_market['yes_token_id']
    
    print(f"Market: {sample_market['question'][:70]}...")
    print(f"Token ID: {token_id[:40]}...")
    print()
    
    try:
        orderbook = pm.fetch_orderbook(token_id)
        
        if len(orderbook) > 0:
            print(f"ORDER BOOK ({len(orderbook)} levels)")
            print("=" * 50)
            
            bids = orderbook[orderbook['side'] == 'bid'].sort_values('price', ascending=False)
            asks = orderbook[orderbook['side'] == 'ask'].sort_values('price', ascending=True)
            
            print(f"\nBIDS (buyers): {len(bids)} levels")
            if len(bids) > 0:
                print(bids[['price', 'size']].head(5))
            
            print(f"\nASKS (sellers): {len(asks)} levels")
            if len(asks) > 0:
                print(asks[['price', 'size']].head(5))
            
            # Calculate spread
            if len(bids) > 0 and len(asks) > 0:
                best_bid = bids['price'].max()
                best_ask = asks['price'].min()
                spread = best_ask - best_bid
                print(f"\nSpread: {spread:.4f} ({spread*100:.2f}%)")
        else:
            print("No orderbook data (market may be resolved or illiquid)")
    except Exception as e:
        print(f"Could not fetch orderbook: {e}")
else:
    print("No markets with token IDs found")

## 5. Database - Load All Markets

Query markets stored in the local SQLite database.

In [ ]:
# Load all markets from database (including resolved)
df = pm.load_markets(limit=500, active_only=False)

print(f"Loaded {len(df)} markets from database")
print(f"\nColumns: {list(df.columns)}")
print()
df[['question', 'yes_price', 'volume', 'active']].head(10)

## 6. Top Markets by Volume

Find the highest volume markets in the database.

In [ ]:
top_markets = pm.load_top_markets(limit=15)

print(f"TOP {len(top_markets)} MARKETS BY VOLUME")
print("=" * 80)
print()

# Format volume as currency
top_markets['volume_fmt'] = top_markets['volume'].apply(lambda x: f"${x:,.0f}")
top_markets[['question', 'volume_fmt', 'yes_price', 'liquidity']]

## 7. Search Markets

Search for markets containing specific keywords.

In [ ]:
# Search for different topics
search_terms = ['Trump', 'election', 'Bitcoin', 'NBA']

for term in search_terms:
    results = pm.search(term, limit=5)
    print(f"\n'{term}': {len(results)} markets found")
    if len(results) > 0:
        for _, row in results.head(3).iterrows():
            print(f"  - {row['question'][:70]}...")

---

## 8. NBA Markets Deep Dive

Explore all NBA-related markets in the database.

In [ ]:
# Search for NBA markets
nba_markets = pm.search('NBA', limit=100)
print(f"Found {len(nba_markets)} markets with 'NBA'")

# Also search for team names
teams = ['Lakers', 'Warriors', 'Celtics', 'Bulls', 'Heat', 'Bucks', 'Nuggets']
all_basketball = nba_markets.copy()

for team in teams:
    team_markets = pm.search(team, limit=50)
    # Add any new markets not already in our list
    for _, row in team_markets.iterrows():
        if row['id'] not in all_basketball['id'].values:
            all_basketball = pd.concat([all_basketball, team_markets[team_markets['id'] == row['id']]])

print(f"Total basketball-related markets: {len(all_basketball)}")

In [ ]:
# Display NBA markets sorted by volume
nba_sorted = all_basketball.sort_values('volume', ascending=False)

print("NBA/BASKETBALL MARKETS (by volume)")
print("=" * 80)
nba_sorted[['question', 'yes_price', 'volume']].head(20)

In [ ]:
# Count markets by team
team_counts = {}
teams_to_check = ['Lakers', 'Warriors', 'Celtics', 'Bulls', 'Heat', 'Bucks', 
                  'Nuggets', 'Suns', 'Mavericks', 'Clippers', '76ers', 'Nets',
                  'Knicks', 'Raptors', 'Jazz', 'Trail Blazers']

for team in teams_to_check:
    count = len(pm.search(team, limit=100))
    if count > 0:
        team_counts[team] = count

# Sort and display
team_df = pd.DataFrame(list(team_counts.items()), columns=['Team', 'Markets'])
team_df = team_df.sort_values('Markets', ascending=False)

print("MARKETS BY NBA TEAM")
print("=" * 30)
print(team_df.to_string(index=False))

## 9. Data Analysis Examples

In [ ]:
# Load all markets for analysis
all_markets = pm.load_markets(limit=1000, active_only=False)

print("MARKET STATISTICS")
print("=" * 50)
print(f"Total markets: {len(all_markets)}")
print(f"Total volume: ${all_markets['volume'].sum():,.0f}")
print(f"Average volume: ${all_markets['volume'].mean():,.0f}")
print(f"Median volume: ${all_markets['volume'].median():,.0f}")
print(f"\nPrice distribution:")
print(f"  Mean yes_price: {all_markets['yes_price'].mean():.2%}")
print(f"  Markets near 50/50: {len(all_markets[(all_markets['yes_price'] > 0.4) & (all_markets['yes_price'] < 0.6)])}")

In [ ]:
# Volume distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume distribution (log scale)
ax1 = axes[0]
volumes = all_markets[all_markets['volume'] > 0]['volume']
ax1.hist(volumes, bins=50, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Volume ($)')
ax1.set_ylabel('Count')
ax1.set_title('Market Volume Distribution')
ax1.set_xscale('log')

# Price distribution
ax2 = axes[1]
prices = all_markets['yes_price']
ax2.hist(prices, bins=20, edgecolor='black', alpha=0.7, color='green')
ax2.set_xlabel('Yes Price (Probability)')
ax2.set_ylabel('Count')
ax2.set_title('Price Distribution')
ax2.axvline(x=0.5, color='red', linestyle='--', label='50%')
ax2.legend()

plt.tight_layout()
plt.show()

## 10. Data Collection

Trigger data collection directly from the notebook to refresh the database.

In [ ]:
# Uncomment to collect fresh data:
# result = pm.collect_now(limit=200)
# print("Collection Result:")
# for key, value in result.items():
#     print(f"  {key}: {value}")

## 11. Direct Database Access (Advanced)

For advanced queries, you can access the database directly using SQLAlchemy.

In [ ]:
from cuic_quant.database import get_engine, get_session, MarketSnapshot
from sqlalchemy import select, func

engine = get_engine()

# Example: Custom query for high-volume NBA markets
with get_session(engine) as session:
    stmt = (
        select(MarketSnapshot)
        .where(
            MarketSnapshot.question.ilike('%Lakers%') |
            MarketSnapshot.question.ilike('%Warriors%') |
            MarketSnapshot.question.ilike('%Celtics%')
        )
        .where(MarketSnapshot.volume > 10000)
        .order_by(MarketSnapshot.volume.desc())
        .limit(10)
    )
    
    results = session.execute(stmt).scalars().all()
    
    print(f"HIGH-VOLUME LAKERS/WARRIORS/CELTICS MARKETS")
    print("=" * 60)
    for m in results:
        print(f"${m.volume:>12,.0f} | {m.question[:50]}...")

In [ ]:
# Aggregation query: Total volume by market status
with get_session(engine) as session:
    stmt = (
        select(
            MarketSnapshot.active,
            func.count(MarketSnapshot.id).label('count'),
            func.sum(MarketSnapshot.volume).label('total_volume')
        )
        .group_by(MarketSnapshot.active)
    )
    
    results = session.execute(stmt).all()
    
    print("VOLUME BY MARKET STATUS")
    print("=" * 50)
    for active, count, volume in results:
        status = "Active" if active else "Resolved"
        print(f"{status:10} | {count:5} markets | ${volume:>15,.0f}")

---

## Summary

This notebook demonstrated:

| Feature | Method | Description |
|---------|--------|-------------|
| **Live API** | `pm.fetch_markets()` | Real-time market data |
| **Order Books** | `pm.fetch_orderbook()` | Bid/ask levels |
| **Database Load** | `pm.load_markets()` | Historical data |
| **Top Markets** | `pm.load_top_markets()` | Sorted by volume |
| **Search** | `pm.search()` | Keyword filtering |
| **Statistics** | `pm.stats()` | Database overview |
| **Collection** | `pm.collect_now()` | Trigger data refresh |
| **Direct SQL** | `get_session()` | Custom queries |

### CLI Commands

```bash
cuic-quant init-db          # Initialize database
cuic-quant collect --limit 500   # Collect data
cuic-quant serve --port 8000     # Start API server
cuic-quant scheduler             # Background collection
cuic-quant stats                 # Show statistics
```